# 資料前處理


---

## 一、 原始感測資料輸入與模擬邏輯 (Sensor Data Simulation Logic)

保留的核心特徵依照其物理特性，設計了專屬的動態模擬邏輯，以高度還原真實產線的設備老化、突發異常與環境變化：

### 1. 環境與設備健康度 (Background / Time-Series)
| 感測特徵 | 欄位名稱 | 單位 | 物理模擬邏輯設計 |
| :--- | :--- | :--- | :--- |
| **濾網壓差** | `filter_diff_pressure_bar` | `bar` | **線性上升 + 突發異常**：每日早上 08:00 定保歸零至 0.1 bar，隨批次微幅上升。內建 1/900 機率觸發嚴重堵塞突波。 |
| **伺服馬達負載** | `servo_torque_load_pct` | `%` | **極緩慢線性遞增**：以總生產批次 (`total_batches`) 為基底，模擬軸承老化斜率，並偶發性疊加極端雜訊。 |
| **空氣壓力** | `air_pressure_bar` | `bar` | **穩定輸出**：設定基準值 3.2 bar，並疊加微小的常態分佈雜訊。 |
| **環境溫度** | `temperature_c` | `°C` | **恆溫空調房**：維持 25.0 ± 0.3°C，排除戶外日夜溫差干擾。 |
| **環境濕度** | `humidity_rh` | `%` | **穩定除濕**：維持 50% ± 1%，確保漆料揮發環境穩定。 |

### 2. 製程與系統狀態標記 (Process & System Event)
| 感測特徵 / 標記 | 欄位名稱 | 單位 | 物理 / 系統邏輯設計 |
| :--- | :--- | :--- | :--- |
| **塗料流量** | `paint_flow_ml_min` | `ml/min` | 隨歷史生產總批次呈 **微幅線性遞減**。 |
| **噴幅寬度** | `spray_width_mm` | `mm` | 隨歷史生產總批次呈 **微幅線性遞減**。 |
| **軌跡追隨誤差** | `path_error_mm` | `mm` | 隨歷史生產總批次（機構老化）呈 **微幅線性遞增**。 |
| **資料品質標記** | `data_quality_flag` | `N/A` | **邊緣端即時賦值**：正常通訊標記為 `normal`；斷線空值經前值填補者標記為 `interpolated`；捕捉到極端雜訊突波則標記為 `outlier`。 |

---

## 二、 感測器門檻值定義 (Thresholds Definitions)
為了提供後端 Rule Engine 觸發警報以及 AI 團隊判定品質基準，本模組制訂了以下各項感測輸入的正常 (Normal)、警告 (Warning) 與異常 (Alarm) 判定區間：

| 欄位名稱 (Sensor) | 正常區間 (Normal) | 警告區間 (Warning) | 異常區間 (Alarm) |
| :--- | :--- | :--- | :--- |
| `filter_diff_pressure_bar` | `0.1 ~ 0.5` | `0.5 ~ 0.7` | `> 0.7` |
| `servo_torque_load_pct` | `40.0 ~ 60.0` | `60.0 ~ 75.0` | `> 75.0` |
| `air_pressure_bar` | `3.0 ~ 3.5` | `< 3.0 或 > 3.5` | `< 2.7 或 > 3.8` |
| `temperature_c` | `20.0 ~ 30.0` | `15.0~20.0` / `30.0~35.0` | `< 15.0 或 > 35.0` |
| `humidity_rh` | `45.0 ~ 65.0` | `35.0~45.0` / `65.0~75.0` | `< 35.0 或 > 75.0` |
| `paint_flow_ml_min` | `105.0 ~ 125.0` | `95.0 ~ 105.0` | `< 95.0 或 > 125.0` |
| `spray_width_mm` | `各站基準值 ± 5.0` <br> *(S1=120, S2=100, S3=82)* | `各站基準值 ± 10.0` | `< 基準值 - 10.0` 或<br> `> 基準值 + 10.0` |
| `path_error_mm` | `0.00 ~ 0.10` | `0.10 ~ 0.15` | `> 0.15` |

---

## 三、 資料流清洗與建構步驟 (Data Pipeline Steps)

本模組捨棄傳統的 CSV 落地交換，改採**原生 SDK 串流直寫 (Direct DB Injection)**，並內建真實產線的狀態機，將底層帶有雜訊的高頻訊號轉為結構化數據：

### Step 1: 斷點續傳與狀態記憶 (State Recovery)
系統啟動時，會自動向 PostgreSQL 查詢最後一筆完工紀錄與總生產批次 (`MAX(start_time)` 與 `total_batches`)。確保系統重啟時能無縫接軌時間軸，且設備「老化曲線」不會因斷電而瞬間重置。

### Step 2: 數位雙生與頻率解耦 (Physical Twin & Frequency Decoupling)
建立站點之間的移位暫存器（模擬輸送帶物理延遲），並將感測頻率徹底解耦：
1. **[慢齒輪] 環境快照 (1/20Hz)**：每 3 分鐘定時採集廠房溫濕度，無論機台是否運作皆持續記錄。
2. **[快齒輪] 高頻動態 (1Hz)**：僅針對「當下有工單停靠」的站點，擷取 50 秒的高頻噴塗動態訊號，絕不在機台空轉時產生垃圾資料。

### Step 3: 班表排程與自動清線 (Shift & Line Clearance)
嚴格執行 `08:00~12:00` 與 `13:00~15:00` 進件班表。到達休息時間時停止進件，但產線**不斷電**，直到管線內所有在製品（WIP）皆跑完 `Station_3` 完成噴塗後，系統才進入休眠，達到真實工廠「零報廢、零卡線」的標準作業流程。

---

## 四、 最終產出 (Deliverables)

模組執行後，不再產出實體靜態檔案，而是透過**微批次寫入 (Micro-batching API)**，將清洗與標記完成的數據拋轉至下游的 **PostgreSQL 資料庫** 中。這種機制將 1Hz 訊號暫存於 Edge 記憶體，每分鐘僅連線寫入一次，完美解決了高頻 I/O 塞爆資料庫的痛點：

* **`batch_run` (批次工單表)**：記錄每張工單的 `batch_id`、啟動時間與完工狀態 (`status`)。將時間抽離獨立成表，避免時序資料重複帶上起迄時間，大幅節省儲存空間。
```json
{
  "batch_id": "B_20260609_080100",
  "batch_start_time": "2026-06-09 08:01:00",
  "batch_end_time": "2026-06-09 08:05:50",
  "status": "ok"
}

sensor_readings (高頻機台資料表)：儲存 1Hz (每秒一筆) 的機台噴塗動態特徵。系統會在 Edge 端暫存 50 秒的高頻訊號，於該分鐘結尾時「整包一次性拋轉」入庫。
{
  "ts": "2026-06-09 08:01:25",
  "batch_id": "B_20260609_080100",
  "station_id": "Station_1",
  "paint_flow_ml_min": 114.85,
  "filter_diff_pressure_bar": 0.102,
  "servo_torque_load_pct": 45.12,
  "air_pressure_bar": 3.21,
  "spray_width_mm": 119.80,
  "path_error_mm": 0.021,
  "data_quality_flag": "normal"
}

sensor_3min_readings (低頻環境資料表)：儲存廠務端穩定的溫濕度環境監控數據。降頻至每 3 分鐘一筆，即使產線休息、無工單時（batch_id 為 null），依然忠實記錄廠房恆溫空調狀態。
{
  "ts": "2026-06-09 08:03:00",
  "batch_id": "B_20260609_080100",
  "station_id": "Station_1",
  "temperature_c": 25.12,
  "humidity_rh": 50.45,
  "data_quality_flag": "normal"
}
```

In [ ]:
import time
import math
from datetime import datetime, timedelta
import numpy as np
import sprayline_db_queries as db

STATION_CONFIG = {
    "Station_1": {"spray_width_base": 120.0},
    "Station_2": {"spray_width_base": 100.0},
    "Station_3": {"spray_width_base": 82.0}
}

def run_physical_twin_pipeline():
    print("啟動邊緣運算【1Hz微批次寫入、15:00清線休眠與斷點記憶】...\n")
    
    try:
        conn = db.get_connection()
        print("成功連線至 PostgreSQL 資料庫\n")
    except Exception as e:
        print(f"資料庫連線失敗: {e}")
        return

    # 模擬器快進速度 (0.01 秒代表現實的 1 秒)
    SPEED_MULTIPLIER = 0.01 
    
    # =================================================
    # 斷點續傳模組 (State & Time Recovery)
    # =================================================
    print("正在與資料庫同步狀態，尋找上次斷點與機台耗損度...")
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT MAX(start_time) FROM batch_run;")
            last_db_time = cur.fetchone()[0]

            if last_db_time:
                virtual_time = last_db_time + timedelta(minutes=1)
                virtual_time = virtual_time.replace(second=0, microsecond=0)
                print(f"時間同步成功！從 {virtual_time.strftime('%Y-%m-%d %H:%M:%S')} 繼續模擬。")
            else:
                virtual_time = datetime(2026, 6, 9, 8, 0, 0)
                print(f"資料庫為空，從初始時間 {virtual_time.strftime('%Y-%m-%d %H:%M:%S')} 開始全新模擬。")

            cur.execute("SELECT COUNT(*) FROM batch_run WHERE status = 'ok';")
            total_batches = cur.fetchone()[0]

            cur.execute("SELECT COUNT(*) FROM batch_run WHERE status = 'ok' AND start_time::date = %s;", (virtual_time.date(),))
            daily_batches = cur.fetchone()[0]

            print(f"耗損同步成功！目前累積總生產 {total_batches} 批，今日已生產 {daily_batches} 批。\n")

    except Exception as e:
        print(f"同步狀態失敗 ({e})，使用預設初始值。")
        virtual_time = datetime(2026, 6, 9, 8, 0, 0)
        total_batches = 0
        daily_batches = 0

    # =================================================
    
    # 1Hz 高頻資料暫存區 (Micro-batch Buffer)
    high_freq_buffers = {"Station_1": [], "Station_2": [], "Station_3": []}
    
    factory_state = {
        "Station_1": None,
        "Conv_1_to_2": [None, None],
        "Station_2": None,
        "Conv_2_to_3": [None, None],
        "Station_3": None
    }
    
    filter_anomaly_timers = {"Station_1": 0, "Station_2": 0, "Station_3": 0}

    while True:
        current_hour = virtual_time.hour
        minute = virtual_time.minute
        sec = virtual_time.second
        
        # 每日早上 08:00 廠務定保
        if current_hour == 8 and minute == 0 and sec == 0:
            daily_batches = 0
            for st in filter_anomaly_timers:
                filter_anomaly_timers[st] = 0
            print(f"[{virtual_time.strftime('%H:%M:%S')}] 🔧 廠務保養：已更換全新濾網，壓差歸零！")

        # 15:00 下班進件班表
        accept_new_batch = (8 <= current_hour < 12) or (13 <= current_hour < 15)
        has_wip = not (
            factory_state["Station_1"] is None and
            all(b is None for b in factory_state["Conv_1_to_2"]) and
            factory_state["Station_2"] is None and
            all(b is None for b in factory_state["Conv_2_to_3"]) and
            factory_state["Station_3"] is None
        )

        # =================================================
        # 產線清空後休眠快轉
        # =================================================
        if not accept_new_batch and not has_wip:
            if 12 <= current_hour < 13:
                next_start = virtual_time.replace(hour=13, minute=0, second=0)
            else:
                next_start = (virtual_time + timedelta(days=1)).replace(hour=8, minute=0, second=0)
            
            print(f"\n[{virtual_time.strftime('%H:%M:%S')}] 產線淨空，系統休眠。時間快轉至 {next_start.strftime('%m-%d %H:%M:%S')}...\n")
            virtual_time = next_start
            time.sleep(1.0)
            continue

        # =================================================
        # [低頻環境快照] 每 3 分鐘紀錄 1 筆 (第 0 秒時觸發)
        # =================================================
        if sec == 0 and minute % 3 == 0:
            env_readings = []
            for station in STATION_CONFIG.keys():
                env_readings.append({
                    "ts": virtual_time,
                    "batch_id": factory_state[station],
                    "station_id": station,
                    "temperature_c": round(np.random.normal(25.0, 0.3), 2),
                    "humidity_rh": round(np.random.normal(50.0, 1.0), 2),
                    "data_quality_flag": "normal"
                })
            try:
                db.insert_sensor_3min_readings_batch(conn, env_readings)
                conn.commit()
            except:
                pass

        # =================================================
        # 物理推移與工單派發 (每分鐘的第 0 秒)
        # =================================================
        if sec == 0:
            # 濾網異常判斷
            for st in STATION_CONFIG.keys():
                if filter_anomaly_timers[st] > 0:
                    filter_anomaly_timers[st] -= 1
                    if filter_anomaly_timers[st] == 0:
                        print(f"[{virtual_time.strftime('%H:%M:%S')}] 🟢 {st} 濾網異常已排除，壓差恢復正常。")
                else:
                    if accept_new_batch and np.random.rand() < (1.0 / 900.0):
                        filter_anomaly_timers[st] = 3 
                        print(f"[{virtual_time.strftime('%H:%M:%S')}] 🚨 警告！{st} 發生濾網突發性嚴重堵塞！")

            # 輸送帶推移
            factory_state["Station_3"] = factory_state["Conv_2_to_3"].pop(-1)
            factory_state["Conv_2_to_3"].insert(0, factory_state["Station_2"])
            factory_state["Station_2"] = factory_state["Conv_1_to_2"].pop(-1)
            factory_state["Conv_1_to_2"].insert(0, factory_state["Station_1"])
            
            if accept_new_batch:
                new_batch_id = f"B_{virtual_time.strftime('%Y%m%d_%H%M%S')}"
                factory_state["Station_1"] = new_batch_id
                try:
                    db.insert_batch_run(conn, batch_id=new_batch_id, start_time=virtual_time, status="running")
                    conn.commit()
                except:
                    pass
            else:
                factory_state["Station_1"] = None

            # 終端機狀態列印 (每分鐘印一次產線狀態)
            if has_wip or accept_new_batch:
                s1_str = factory_state['Station_1'] or '空'
                s2_str = factory_state['Station_2'] or '空'
                s3_str = factory_state['Station_3'] or '空'
                print(f"[{virtual_time.strftime('%H:%M:%S')}] 🏭 狀態: S1({s1_str}) ➡ S2({s2_str}) ➡ S3({s3_str})")

        # =================================================
        # 1Hz 高頻訊號生成與暫存 (第 0~49 秒)
        # =================================================
        if 0 <= sec < 50:
            for station, config in STATION_CONFIG.items():
                current_batch = factory_state[station]
                if current_batch is None:
                    continue
                
                if filter_anomaly_timers[station] > 0:
                    filter_pressure = round(np.random.normal(0.78, 0.04), 3)
                else:
                    filter_pressure = round(np.random.normal(0.10 + (daily_batches * 0.0008), 0.01), 3)

                base_torque = 45.0 + (total_batches * 0.005)
                paint_flow = 115.0 - (total_batches * 0.002)
                spray_width = config["spray_width_base"] - (total_batches * 0.001)
                path_error = 0.02 + (total_batches * 0.00005)

                flag = "normal"
                torque_reading = round(np.random.normal(base_torque, 1.5), 2)
                
                # 極端雜訊
                if np.random.rand() > 0.98:
                    torque_reading = 85.0
                    flag = "outlier"
                    
                reading = {
                    "ts": virtual_time,
                    "batch_id": current_batch,
                    "station_id": station,
                    "paint_flow_ml_min": round(np.random.normal(paint_flow, 0.5), 2),
                    "filter_diff_pressure_bar": filter_pressure,
                    "servo_torque_load_pct": torque_reading,
                    "air_pressure_bar": round(np.random.normal(3.2, 0.05), 2), 
                    "spray_width_mm": round(np.random.normal(spray_width, 0.5), 2),
                    "path_error_mm": round(np.random.normal(path_error, 0.005), 3),
                    "data_quality_flag": flag
                }
                # 存入記憶體 Buffer，還不寫入 DB
                high_freq_buffers[station].append(reading)

        # =================================================
        # 微批次寫入 (第 50 秒結案拋轉 DB)
        # =================================================
        if sec == 50:
            all_readings = []
            # 把三個站台的 Buffer 合併成一大包
            for station in STATION_CONFIG.keys():
                all_readings.extend(high_freq_buffers[station])
                
            try:
                if all_readings:
                    # 整包拋轉給 DB，大幅減少連線次數
                    db.insert_sensor_readings_batch(conn, all_readings)
                
                completed_batch = factory_state["Station_3"]
                if completed_batch:
                    db.update_batch_status(conn, batch_id=completed_batch, status="ok", ended_time=virtual_time)
                    total_batches += 1
                    daily_batches += 1
                
                conn.commit()
            except Exception as e:
                conn.rollback()
            
            # 拋轉完畢，清空 Buffer 迎接下一分鐘
            for station in STATION_CONFIG.keys():
                high_freq_buffers[station].clear()

        # 時間每一秒往前推移
        virtual_time += timedelta(seconds=1)
        time.sleep(SPEED_MULTIPLIER)

if __name__ == "__main__":
    run_physical_twin_pipeline()